# 03 — Late-Risk Classification

This notebook reframes the problem from ETA prediction to risk prediction: what is the probability that this airport taxi trip will be unusually slow?


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

DATA = Path('../data/processed/taxi_clean_for_modeling.csv')
df = pd.read_csv(DATA, parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])
if 'duration_min' not in df.columns:
    df['duration_min'] = df['trip_duration_min']
if 'is_late' not in df.columns:
    median_dur = df.groupby(['airport','pickup_hour','pickup_dow'])['duration_min'].median().rename('typical_duration_min').reset_index()
    df = df.merge(median_dur, on=['airport','pickup_hour','pickup_dow'], how='left')
    df['is_late'] = (df['duration_min'] > 1.2 * df['typical_duration_min']).astype(int)

df['is_late'].value_counts(normalize=True)

## Why Accuracy Is Misleading

Late trips are the minority class. A model can look accurate by predicting almost every trip as on-time, but that fails the business goal of catching risky airport rides.


In [ ]:
features = ['trip_distance', 'passenger_count', 'pickup_hour', 'pickup_dow', 'payment_type', 'airport', 'PULocationID']
target = 'is_late'

df = df.sort_values('tpep_pickup_datetime').reset_index(drop=True)
X = df[features]
y = df[target]

split_idx = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

cat_cols = ['airport', 'PULocationID', 'payment_type']
num_cols = ['trip_distance', 'passenger_count', 'pickup_hour', 'pickup_dow']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols),
])

In [ ]:
def cls_metrics(y_true, proba, threshold=0.50):
    pred = (proba >= threshold).astype(int)
    return {
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, proba),
    }

## Logistic Regression Baseline

This gives an interpretable baseline before tree-based classifiers.


In [ ]:
logit = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
logit.fit(X_train, y_train)
logit_proba = logit.predict_proba(X_test)[:, 1]
cls_metrics(y_test, logit_proba, threshold=0.50)

## Random Forest Classifier

The full project also tested CatBoost, which performed best on ROC-AUC and threshold tradeoff.


In [ ]:
rf = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
cls_metrics(y_test, rf_proba, threshold=0.50)

## Threshold Tuning

The same probability model can support different policies. A traveler may want high recall and accept false alarms; an operations team may prefer fewer alerts.


In [ ]:
thresholds = [0.50, 0.40, 0.35, 0.30, 0.25, 0.20]
results = pd.DataFrame([cls_metrics(y_test, rf_proba, t) for t in thresholds])
results

## Portfolio Result Summary

The full tuned CatBoost classifier achieved **ROC-AUC ≈ 0.725**. At a risk-averse threshold of **0.40**, recall reached approximately **0.826**, catching most late trips at the cost of more false positives.
